# Python `Enum` Aliases — Advanced Problems as a Step-by-Step Tutorial

**A new set of 18 worked problems, organized as a tutorial with short explanations between runnable code cells.**

This notebook takes the supplied lesson’s teaching pattern—explain one idea, run a tiny example, interpret the result, then extend it—into advanced API design, implementation traps, protocol compatibility, and auditability. Its problems and examples are newly written rather than a copy of the earlier problem set.

**Prerequisites:** comfortable with classes, dictionaries, functions, and exceptions. **Environment:** Python 3.10+; only the standard library is used. Use **Kernel → Restart & Run All** to follow the intended order.

Every problem includes a concrete scenario, successive investigations, a working solution, an edge case, assertions, and a further challenge. Assertion cells test behavior; the prose explains *why* it holds.

## Orientation — map the source lesson to this notebook

The provided lesson introduces four core facts: equal values can create aliases; the first member is canonical (called the “master” in the lesson); name lookup and value lookup return that same member; ordinary iteration omits aliases whereas `__members__` retains every declared name. It then shows status aliases and the `@enum.unique` safeguard.

**Important boundary:** built-in aliases are *additional Python member names*, not alternative runtime values. For an enum whose `RUNNING`, `BUSY`, and `PROCESSING` names all have the value `2`, `State['BUSY']` succeeds but `State('busy')` does not. The latter requires a parser or a value-mapping policy. Many advanced problems below begin with this distinction.

**Best-practice rule used throughout:** store unique semantic states in an enum; preserve source spellings separately when they matter for compatibility, diagnostics, warnings, or round trips.

In [1]:
import enum
import copy
import json
import pickle
import warnings
from collections import Counter
from dataclasses import dataclass
from typing import Mapping

print('Tutorial environment ready')

Tutorial environment ready


### A 30-second reference example

Run this once to establish the vocabulary. The name `ACTIVE` is canonical because it is defined before `BUSY`. `__members__` includes both spellings, but `list(...)` includes each distinct enum member only once.

In [2]:
class ReferenceState(enum.Enum):
    ACTIVE = 1
    BUSY = 1
    DONE = 2

assert ReferenceState.ACTIVE is ReferenceState.BUSY
assert ReferenceState['BUSY'].name == 'ACTIVE'
assert ReferenceState(1) is ReferenceState.ACTIVE
assert [member.name for member in ReferenceState] == ['ACTIVE', 'DONE']
assert list(ReferenceState.__members__) == ['ACTIVE', 'BUSY', 'DONE']
print('Canonical names:', [member.name for member in ReferenceState])
print('All declared names:', list(ReferenceState.__members__))

Canonical names: ['ACTIVE', 'DONE']
All declared names: ['ACTIVE', 'BUSY', 'DONE']


---

# Part I — Alias formation and class construction

These problems go beyond checking identity: they reveal *when* an alias forms and what information is discarded.

## Problem 01 — Treat declaration order as an API decision

**Scenario.** An API generator assembles an enum from a partner’s ordered code table.

**Your task.** Create the enum dynamically, reverse two equal-valued names, and explain why the externally displayed name changes even though the numerical protocol is unchanged.

**How to work through this tutorial:** predict the result before running each investigation cell. Read the interpretation before moving to the implementation. The last code cell is an executable solution check.

### Step 1 — Identify the invariant

Functional construction obeys the same first-name-wins rule as a class statement. Compare two ordered input tables that use the same codes.

In [3]:
p01_old_spec = [('ECONOMY', 10), ('STANDARD', 10), ('PRIORITY', 20)]
p01_new_spec = [('STANDARD', 10), ('ECONOMY', 10), ('PRIORITY', 20)]
P01Old = enum.Enum('P01Old', p01_old_spec)
P01New = enum.Enum('P01New', p01_new_spec)
print(P01Old(10), P01New(10))

P01Old.ECONOMY P01New.STANDARD


**What the observation tells us.** A numeric lookup resolves to whichever name appeared first in that enum’s definition; a member does not remember the spelling used for the lookup.

In [4]:
print(P01Old['STANDARD'].name, P01New['ECONOMY'].name)
print(tuple(P01Old.__members__), tuple(P01New.__members__))

ECONOMY STANDARD
('ECONOMY', 'STANDARD', 'PRIORITY') ('STANDARD', 'ECONOMY', 'PRIORITY')


### Step 2 — Construct the solution

Build a contract snapshot that records canonical names by numeric value. This is the piece a reviewer should compare before releasing a generator change.

In [5]:
def p01_contract(spec):
    cls = enum.Enum('GeneratedContract', spec)
    return {member.value: member.name for member in cls}

p01_before = p01_contract(p01_old_spec)
p01_after = p01_contract(p01_new_spec)
print('Before:', p01_before, 'After:', p01_after)

Before: {10: 'ECONOMY', 20: 'PRIORITY'} After: {10: 'STANDARD', 20: 'PRIORITY'}


### Step 3 — Apply it to an edge case

Check name-based input separately from canonical name-based output. Both accepted names still exist, despite the output difference.

In [6]:
p01_name_inputs = ['ECONOMY', 'STANDARD']
for name in p01_name_inputs:
    print(name, '->', P01New[name].name, '->', P01New[name].value)

ECONOMY -> STANDARD -> 10
STANDARD -> STANDARD -> 10


### Full solution: reasoning and correctness

Both tables have the same accepted names and values, but distinct canonical output at code 10. Thus declaration ordering can be a serialization compatibility concern.

**Verification.** The following cell uses assertions. If it runs without error, the stated behavior holds in this notebook environment.

In [7]:
assert P01Old(10).name == 'ECONOMY'
assert P01New(10).name == 'STANDARD'
assert set(P01Old.__members__) == set(P01New.__members__)
assert p01_before != p01_after
assert P01Old['STANDARD'].value == P01New['STANDARD'].value == 10
print('P01 passed')

P01 passed


**Takeaway / further challenge.** Add a release check that rejects a changed canonical-name snapshot unless a migration flag is supplied.

## Problem 02 — Expose a tuple-value metadata trap

**Scenario.** A designer puts a code and a display label together in each enum value.

**Your task.** Demonstrate how repeated tuples collapse into an alias, then redesign the type so display text belongs to its own mapping.

**How to work through this tutorial:** predict the result before running each investigation cell. Read the interpretation before moving to the implementation. The last code cell is an executable solution check.

### Step 1 — Identify the invariant

Enum compares the *whole value* for equality. Two equal tuples are not separate members just because they have different names.

In [8]:
class P02Flawed(enum.Enum):
    QUEUED = ('Q', 'In queue')
    WAITING = ('Q', 'In queue')
    SENT = ('S', 'Sent')

print(P02Flawed.WAITING is P02Flawed.QUEUED)
print(P02Flawed.WAITING.name, P02Flawed.WAITING.value)

True
QUEUED ('Q', 'In queue')


**What the observation tells us.** Python retains one enum member object per distinct value. Equal tuple payloads do not provide a place to store independent alias metadata.

In [9]:
print('Declared spellings:', list(P02Flawed.__members__))
print('Iterated names:', [x.name for x in P02Flawed])

Declared spellings: ['QUEUED', 'WAITING', 'SENT']
Iterated names: ['QUEUED', 'SENT']


### Step 2 — Construct the solution

Use one stable code as the enum value, and put an independent label policy in a mapping. Names that must be accepted can be explicit aliases.

In [10]:
@enum.unique
class P02State(enum.Enum):
    QUEUED = 'Q'
    SENT = 'S'

p02_labels = {P02State.QUEUED: 'In queue', P02State.SENT: 'Sent'}
p02_external_names = {'QUEUED': P02State.QUEUED,
                      'WAITING': P02State.QUEUED,
                      'SENT': P02State.SENT}

def p02_display(external_name):
    state = p02_external_names[external_name]
    return {'input': external_name, 'code': state.value,
            'label': p02_labels[state]}
print(p02_display('WAITING'))

{'input': 'WAITING', 'code': 'Q', 'label': 'In queue'}


### Step 3 — Apply it to an edge case

If a partner needs a different label for `WAITING`, the raw token is available. Storing it only as an enum member would make that distinction impossible.

In [11]:
p02_partner_labels = {'WAITING': 'Awaiting dispatch'}
def p02_partner_display(name):
    result = p02_display(name)
    return {**result, 'label': p02_partner_labels.get(name, result['label'])}
print(p02_partner_display('WAITING'))

{'input': 'WAITING', 'code': 'Q', 'label': 'Awaiting dispatch'}


### Full solution: reasoning and correctness

The semantic enum encodes state. The external spelling and partner-specific display text are separate metadata; neither can disappear due to alias identity.

**Verification.** The following cell uses assertions. If it runs without error, the stated behavior holds in this notebook environment.

In [12]:
assert P02Flawed.WAITING is P02Flawed.QUEUED
assert p02_display('WAITING')['code'] == 'Q'
assert p02_partner_display('WAITING')['label'] == 'Awaiting dispatch'
assert len(P02State) == 2
print('P02 passed')

P02 passed


**Takeaway / further challenge.** Try three languages of labels without adding any new enum members.

## Problem 03 — Detect lost fields in a custom `__new__`

**Scenario.** A packet type has a numeric wire code and a human-readable description; two definitions share one wire code.

**Your task.** Show which description survives alias creation and keep alternative descriptions in an external registry.

**How to work through this tutorial:** predict the result before running each investigation cell. Read the interpretation before moving to the implementation. The last code cell is an executable solution check.

### Step 1 — Identify the invariant

For enums with custom `__new__`, assigning `_value_` determines member equality. Multiple definitions with the same `_value_` share a single resulting member.

In [13]:
class P03Packet(enum.Enum):
    def __new__(cls, code, description):
        obj = object.__new__(cls)
        obj._value_ = code
        obj.description = description
        return obj

    CREATE = (100, 'Create a record')
    ADD = (100, 'Add a record')
    DELETE = (200, 'Remove a record')

print(P03Packet.ADD is P03Packet.CREATE)
print(P03Packet.ADD.description)

True
Create a record


**What the observation tells us.** The `ADD` name resolves to the first object, including its `description` attribute. Do not rely on alias definitions to hold differing per-name metadata.

In [14]:
print({name: member.description for name, member in P03Packet.__members__.items()})

{'CREATE': 'Create a record', 'ADD': 'Create a record', 'DELETE': 'Remove a record'}


### Step 2 — Construct the solution

Keep a distinct code enum and key descriptions by *external name*. The alias can remain for callers that expect dot or bracket access.

In [15]:
class P03Code(enum.Enum):
    CREATE = 100
    ADD = 100
    DELETE = 200

p03_descriptions = {'CREATE': 'Create a record',
                    'ADD': 'Add a record',
                    'DELETE': 'Remove a record'}

def p03_describe(name):
    return P03Code[name], p03_descriptions[name]
print(p03_describe('ADD'))

(<P03Code.CREATE: 100>, 'Add a record')


### Step 3 — Apply it to an edge case

Canonical display and alias-sensitive display are different requirements. Decide which one an API needs and write that decision into its interface.

In [16]:
def p03_canonical_description(name):
    member = P03Code[name]
    return p03_descriptions[member.name]
print('Canonical:', p03_canonical_description('ADD'))
print('Source-specific:', p03_describe('ADD')[1])

Canonical: Create a record
Source-specific: Add a record


### Full solution: reasoning and correctness

Both descriptions can now be recovered as appropriate. Previously the enum kept only one description for numeric code 100.

**Verification.** The following cell uses assertions. If it runs without error, the stated behavior holds in this notebook environment.

In [17]:
assert P03Packet.ADD.description == 'Create a record'
assert p03_describe('ADD')[1] == 'Add a record'
assert p03_describe('ADD')[0] is P03Code.CREATE
assert p03_canonical_description('ADD') == 'Create a record'
print('P03 passed')

P03 passed


**Takeaway / further challenge.** Extend `p03_describe` to reject unknown names with a domain-specific exception.

## Problem 04 — Diagnose an accidental `_generate_next_value_` alias

**Scenario.** A developer customizes `auto()` for compact numeric codes and accidentally repeats every generated code.

**Your task.** Prove the alias was caused by the generator, then correct it and enforce uniqueness.

**How to work through this tutorial:** predict the result before running each investigation cell. Read the interpretation before moving to the implementation. The last code cell is an executable solution check.

### Step 1 — Identify the invariant

The generator receives a count of *declared* members, including any earlier aliases. Returning `count // 2` here creates repeated values.

In [18]:
class P04Broken(enum.Enum):
    def _generate_next_value_(name, start, count, last_values):
        return count // 2

    A = enum.auto()
    B = enum.auto()
    C = enum.auto()
    D = enum.auto()

print([(name, member.value) for name, member in P04Broken.__members__.items()])

[('A', 0), ('B', 0), ('C', 1), ('D', 1)]


**What the observation tells us.** Pairs of equal numeric values become aliases. Iteration therefore hides B and D rather than indicating that four distinct generated codes exist.

In [19]:
print('Real members:', list(P04Broken))
print('All names:', list(P04Broken.__members__))

Real members: [<P04Broken.A: 0>, <P04Broken.C: 1>]
All names: ['A', 'B', 'C', 'D']


### Step 2 — Construct the solution

Use a one-to-one generator and `@unique`, so a regression fails during class creation rather than silently merging commands.

In [20]:
@enum.unique
class P04Fixed(enum.Enum):
    def _generate_next_value_(name, start, count, last_values):
        return count + 1

    A = enum.auto()
    B = enum.auto()
    C = enum.auto()
    D = enum.auto()

print([(item.name, item.value) for item in P04Fixed])

[('A', 1), ('B', 2), ('C', 3), ('D', 4)]


### Step 3 — Apply it to an edge case

Be precise: `@unique` checks the final enum, not the mathematical correctness of every generated code. Its guarantee here is just that no aliases remain.

In [21]:
try:
    enum.unique(P04Broken)
except ValueError as exc:
    print('Rejected:', exc)

Rejected: duplicate values found in <enum 'P04Broken'>: B -> A, D -> C


### Full solution: reasoning and correctness

An enum generator is executable policy. Test exact values and the absence of aliases; these are separate contracts.

**Verification.** The following cell uses assertions. If it runs without error, the stated behavior holds in this notebook environment.

In [22]:
assert len(P04Broken) == 2
assert P04Broken.A is P04Broken.B
assert P04Broken.C is P04Broken.D
assert [x.value for x in P04Fixed] == [1, 2, 3, 4]
assert len(P04Fixed.__members__) == len(P04Fixed)
print('P04 passed')

P04 passed


**Takeaway / further challenge.** Try a generator that skips reserved codes while keeping a unique mapping.

## Problem 05 — Preflight external specifications *before* class construction

**Scenario.** A code generator ingests ordered `(name, value)` pairs from a data file.

**Your task.** Reject duplicate names and accidental duplicate values before constructing the enum; permit aliases only when explicitly configured.

**How to work through this tutorial:** predict the result before running each investigation cell. Read the interpretation before moving to the implementation. The last code cell is an executable solution check.

### Step 1 — Identify the invariant

Duplicate names and duplicate values are distinct errors. Repeating a name is invalid; repeating a value is legal aliasing unless your contract forbids it.

In [23]:
p05_good = [('OPEN', 10), ('ACTIVE', 10), ('CLOSED', 20)]
p05_bad = [('OPEN', 10), ('ACTIVE', 10), ('CLOSED', 20), ('OPEN', 30)]
print('Name occurrences:', [name for name, _ in p05_bad])

Name occurrences: ['OPEN', 'ACTIVE', 'CLOSED', 'OPEN']


**What the observation tells us.** Putting the pairs into a dictionary first would overwrite duplicate names and conceal the error. Validate the ordered sequence before discarding information.

In [24]:
p05_overwritten = dict(p05_bad)
print('Dictionary has', len(p05_overwritten), 'entries versus', len(p05_bad), 'input rows')

Dictionary has 3 entries versus 4 input rows


### Step 2 — Construct the solution

The following validator uses equality to detect repeated values, so it also handles values that are not hashable. For this example the supplied values are ordinary integers.

In [25]:
def p05_build(name, rows, *, aliases_allowed):
    used_names = set()
    seen_values = []
    for member_name, value in rows:
        if member_name in used_names:
            raise ValueError(f'duplicate name: {member_name}')
        if not aliases_allowed and any(value == old for old in seen_values):
            raise ValueError(f'duplicate value for: {member_name}')
        used_names.add(member_name)
        seen_values.append(value)
    return enum.Enum(name, rows)

P05Compatible = p05_build('P05Compatible', p05_good, aliases_allowed=True)
print(P05Compatible.ACTIVE is P05Compatible.OPEN)

True


### Step 3 — Apply it to an edge case

Exercise both policy branches and ensure the error message makes the reason visible.

In [26]:
for rows, allowed in [(p05_bad, True), (p05_good, False)]:
    try:
        p05_build('P05ShouldFail', rows, aliases_allowed=allowed)
    except ValueError as exc:
        print(type(exc).__name__, str(exc))

ValueError duplicate name: OPEN
ValueError duplicate value for: ACTIVE


### Full solution: reasoning and correctness

Preflight checks recover errors that would otherwise be masked by premature dictionary conversion, and they distinguish intentional aliases from forbidden collisions.

**Verification.** The following cell uses assertions. If it runs without error, the stated behavior holds in this notebook environment.

In [27]:
assert list(P05Compatible.__members__) == ['OPEN', 'ACTIVE', 'CLOSED']
for rows, allowed in [(p05_bad, True), (p05_good, False)]:
    try:
        p05_build('Nope', rows, aliases_allowed=allowed)
    except ValueError:
        pass
    else:
        raise AssertionError('invalid specification was accepted')
print('P05 passed')

P05 passed


**Takeaway / further challenge.** Add validation that member names are valid public identifiers according to your own style guide.

---

# Part II — Safe control flow

A member alias is the same object. These problems show why dispatch, pattern matching, and transitions need policies about semantic identity versus original spelling.

## Problem 06 — Guard against overwritten command registrations

**Scenario.** Two independently developed plugins register different callbacks for names that represent the same command code.

**Your task.** Show how an enum-keyed dictionary silently overwrites one callback and build a registration function that rejects conflicting implementations.

**How to work through this tutorial:** predict the result before running each investigation cell. Read the interpretation before moving to the implementation. The last code cell is an executable solution check.

### Step 1 — Identify the invariant

An alias can be used as a dictionary key, but it hashes and compares like its canonical member because it *is* that member.

In [28]:
class P06Command(enum.Enum):
    EXECUTE = 1
    RUN = 1
    CANCEL = 2

def p06_primary(): return 'primary'
def p06_legacy(): return 'legacy'

p06_unsafe = {P06Command.EXECUTE: p06_primary, P06Command.RUN: p06_legacy}
print('Entries:', len(p06_unsafe), 'Chosen:', p06_unsafe[P06Command.EXECUTE]())

Entries: 1 Chosen: legacy


**What the observation tells us.** The second assignment overwrites the first. A literal does not raise an exception for duplicate keys, so the registry lost a registration.

In [29]:
assert P06Command.EXECUTE is P06Command.RUN
print('Identity:', P06Command.RUN is P06Command.EXECUTE)

Identity: True


### Step 2 — Construct the solution

Accept an ordered registration sequence and compare existing callback identities before assignment. Identical repeated callbacks are harmless; different ones are a conflict.

In [30]:
def p06_register(rows):
    result = {}
    for name, callback in rows:
        member = P06Command[name]
        if member in result and result[member] is not callback:
            raise ValueError(f'conflicting handler for {member.name}: {name}')
        result[member] = callback
    return result

p06_good = p06_register([('EXECUTE', p06_primary), ('RUN', p06_primary)])
print(p06_good[P06Command.RUN]())

primary


### Step 3 — Apply it to an edge case

Test the rejection path using the original conflicting plugin registration.

In [31]:
try:
    p06_register([('EXECUTE', p06_primary), ('RUN', p06_legacy)])
except ValueError as exc:
    print('Caught collision:', exc)

Caught collision: conflicting handler for EXECUTE: RUN


### Full solution: reasoning and correctness

Identity-safe handlers make duplicate registration explicit instead of letting a dictionary silently pick the last implementation.

**Verification.** The following cell uses assertions. If it runs without error, the stated behavior holds in this notebook environment.

In [32]:
assert len(p06_good) == 1
assert p06_good[P06Command.EXECUTE] is p06_primary
try:
    p06_register([('EXECUTE', p06_primary), ('RUN', p06_legacy)])
except ValueError:
    pass
else:
    raise AssertionError('expected duplicate-handler error')
print('P06 passed')

P06 passed


**Takeaway / further challenge.** Improve the registry to record *all* plugin names that attempted to register each semantic command.

## Problem 07 — Understand alias patterns in `match`

**Scenario.** An event router has two syntactic event names sharing one semantic value.

**Your task.** Predict which `match` branch executes, then design a function that can distinguish the input name when needed.

**How to work through this tutorial:** predict the result before running each investigation cell. Read the interpretation before moving to the implementation. The last code cell is an executable solution check.

### Step 1 — Identify the invariant

Python structural pattern matching compares enum member patterns by value/equality. Aliases cannot produce two distinct branches.

In [33]:
class P07Signal(enum.Enum):
    SUCCESS = 200
    OK = 200
    FAILURE = 500

def p07_naive(signal):
    match signal:
        case P07Signal.SUCCESS:
            return 'success branch'
        case P07Signal.OK:
            return 'alias branch'
        case _:
            return 'other branch'

print(p07_naive(P07Signal.OK))

success branch


**What the observation tells us.** `P07Signal.OK` is identical to `P07Signal.SUCCESS`; the first branch matches. The second branch cannot identify the spelling used by a caller.

In [34]:
print(P07Signal.OK.name)
print(P07Signal['OK'] is P07Signal['SUCCESS'])

SUCCESS
True


### Step 2 — Construct the solution

If the external spelling changes the response, inspect it before or alongside parsing to the enum. Do not turn a protocol spelling distinction into an enum identity distinction.

In [35]:
def p07_route_token(token):
    member = P07Signal[token]
    match member:
        case P07Signal.SUCCESS:
            action = 'acknowledge'
        case P07Signal.FAILURE:
            action = 'escalate'
    return {'action': action, 'source_token': token}

print(p07_route_token('OK'))

{'action': 'acknowledge', 'source_token': 'OK'}


### Step 3 — Apply it to an edge case

A transition that should be identical for both names uses the canonical action; observability and logging use the source token.

In [36]:
for token in ['SUCCESS', 'OK', 'FAILURE']:
    print(token, '->', p07_route_token(token))

SUCCESS -> {'action': 'acknowledge', 'source_token': 'SUCCESS'}
OK -> {'action': 'acknowledge', 'source_token': 'OK'}
FAILURE -> {'action': 'escalate', 'source_token': 'FAILURE'}


### Full solution: reasoning and correctness

The router now differentiates syntax from semantics. It does not pretend alias cases in `match` are independent.

**Verification.** The following cell uses assertions. If it runs without error, the stated behavior holds in this notebook environment.

In [37]:
assert p07_naive(P07Signal.OK) == 'success branch'
assert p07_route_token('SUCCESS')['action'] == p07_route_token('OK')['action']
assert p07_route_token('SUCCESS')['source_token'] != p07_route_token('OK')['source_token']
assert p07_route_token('FAILURE')['action'] == 'escalate'
print('P07 passed')

P07 passed


**Takeaway / further challenge.** Add a third alias and prove that no extra semantic branch is required.

## Problem 08 — Validate a state machine with aliases

**Scenario.** A workflow accepts both `ACTIVE` and `PROCESSING` for the same internal state.

**Your task.** Implement a transition table keyed by semantic enum members and reject illegal edges while preserving the original user-facing token.

**How to work through this tutorial:** predict the result before running each investigation cell. Read the interpretation before moving to the implementation. The last code cell is an executable solution check.

### Step 1 — Identify the invariant

Inspect whether aliases really expand the number of states. They do not: a state machine should normally be indexed by unique semantic states.

In [38]:
class P08State(enum.Enum):
    NEW = 0
    ACTIVE = 1
    PROCESSING = 1
    DONE = 2
    FAILED = 3

print('State count:', len(P08State), 'Accepted names:', len(P08State.__members__))

State count: 4 Accepted names: 5


**What the observation tells us.** Four states exist, but five names are accepted. A transition table needs four semantic keys, not five duplicated rows.

In [39]:
p08_edges = {
    P08State.NEW: {P08State.ACTIVE},
    P08State.ACTIVE: {P08State.DONE, P08State.FAILED},
    P08State.DONE: set(),
    P08State.FAILED: set(),
}
print(p08_edges[P08State.PROCESSING])

{<P08State.FAILED: 3>, <P08State.DONE: 2>}


### Step 2 — Construct the solution

Create a transition function that separates parsing, edge validation, and audit output. Exceptions carry the canonical state for consistent debugging.

In [40]:
def p08_transition(current_name, next_name):
    current = P08State[current_name]
    target = P08State[next_name]
    if target not in p08_edges[current]:
        raise ValueError(f'illegal transition {current.name} -> {target.name}')
    return {'previous': current, 'next': target,
            'source_next_name': next_name}

print(p08_transition('NEW', 'PROCESSING'))

{'previous': <P08State.NEW: 0>, 'next': <P08State.ACTIVE: 1>, 'source_next_name': 'PROCESSING'}


### Step 3 — Apply it to an edge case

Confirm that aliases obey exactly the same transition restrictions, including failure cases.

In [41]:
for start, end in [('ACTIVE', 'DONE'), ('PROCESSING', 'FAILED')]:
    print(p08_transition(start, end))
try:
    p08_transition('DONE', 'PROCESSING')
except ValueError as exc:
    print('Rejected:', exc)

{'previous': <P08State.ACTIVE: 1>, 'next': <P08State.DONE: 2>, 'source_next_name': 'DONE'}
{'previous': <P08State.ACTIVE: 1>, 'next': <P08State.FAILED: 3>, 'source_next_name': 'FAILED'}
Rejected: illegal transition DONE -> ACTIVE


### Full solution: reasoning and correctness

Every accepted name is parsed to a state before validation. A separate field preserves how the caller named the target, without duplicating state logic.

**Verification.** The following cell uses assertions. If it runs without error, the stated behavior holds in this notebook environment.

In [42]:
assert p08_transition('NEW', 'PROCESSING')['next'] is P08State.ACTIVE
assert p08_transition('ACTIVE', 'DONE')['next'] is P08State.DONE
try:
    p08_transition('DONE', 'PROCESSING')
except ValueError:
    pass
else:
    raise AssertionError('illegal transition accepted')
assert len(p08_edges) == len(P08State) == 4
print('P08 passed')

P08 passed


**Takeaway / further challenge.** Add a transition validator that rejects missing semantic keys or duplicated transition descriptions.

## Problem 09 — Prove that aliases cannot represent distinct business events

**Scenario.** An audit system considers `AUTHORIZE` and `CAPTURE` separate business events, but a designer assigns both code 7.

**Your task.** Diagnose the information loss and design an event model that keeps the events distinct while sharing the wire code if necessary.

**How to work through this tutorial:** predict the result before running each investigation cell. Read the interpretation before moving to the implementation. The last code cell is an executable solution check.

### Step 1 — Identify the invariant

An enum member’s identity is determined by its value. Distinct business events must have distinct member values, even if one external field happens to be shared.

In [43]:
class P09Incorrect(enum.Enum):
    AUTHORIZE = 7
    CAPTURE = 7

p09_history = [P09Incorrect.AUTHORIZE, P09Incorrect.CAPTURE]
print([event.name for event in p09_history])

['AUTHORIZE', 'AUTHORIZE']


**What the observation tells us.** Both records now appear as AUTHORIZE. This is not a logging bug: the original distinction was never stored in the enum objects.

In [44]:
print('Distinct semantic events:', len(set(p09_history)))

Distinct semantic events: 1


### Step 2 — Construct the solution

Model semantic events using unique values. Store the shared external code in a separate mapping, where many-to-one relationships are allowed.

In [45]:
@enum.unique
class P09Event(enum.Enum):
    AUTHORIZE = 'authorize'
    CAPTURE = 'capture'

p09_wire_codes = {P09Event.AUTHORIZE: 7, P09Event.CAPTURE: 7}

def p09_record(event):
    return {'event': event.name, 'wire_code': p09_wire_codes[event]}
print([p09_record(e) for e in P09Event])

[{'event': 'AUTHORIZE', 'wire_code': 7}, {'event': 'CAPTURE', 'wire_code': 7}]


### Step 3 — Apply it to an edge case

Observe that decoding a wire code without other information is ambiguous. Make the missing discriminator explicit rather than arbitrarily choosing an event.

In [46]:
def p09_decode_wire(code):
    candidates = [event for event, number in p09_wire_codes.items()
                  if number == code]
    if len(candidates) != 1:
        raise ValueError(f'wire code {code} needs an event discriminator')
    return candidates[0]
try:
    p09_decode_wire(7)
except ValueError as exc:
    print('Ambiguous:', exc)

Ambiguous: wire code 7 needs an event discriminator


### Full solution: reasoning and correctness

A shared code is a property, not a member identity. Unique semantic values prevent two different actions from collapsing into a single event.

**Verification.** The following cell uses assertions. If it runs without error, the stated behavior holds in this notebook environment.

In [47]:
assert P09Incorrect.AUTHORIZE is P09Incorrect.CAPTURE
assert P09Event.AUTHORIZE is not P09Event.CAPTURE
assert p09_record(P09Event.CAPTURE)['wire_code'] == 7
try:
    p09_decode_wire(7)
except ValueError:
    pass
else:
    raise AssertionError('ambiguous code decoded')
print('P09 passed')

P09 passed


**Takeaway / further challenge.** Add a second input field called `operation` to make the wire format unambiguous.

---

# Part III — Persistence and compatibility

Alias behavior becomes much more consequential when names cross a process, version, or storage boundary.

## Problem 10 — Design a stable versioned wire format

**Scenario.** Two application releases reverse which of two equal-valued names is canonical.

**Your task.** Compare name-based and numeric serialization; choose a versioned representation that survives canonical-name changes.

**How to work through this tutorial:** predict the result before running each investigation cell. Read the interpretation before moving to the implementation. The last code cell is an executable solution check.

### Step 1 — Identify the invariant

First reconstruct the compatibility hazard in isolation. Both enums accept the same names; their canonical names differ.

In [48]:
class P10V1(enum.Enum):
    STARTED = 1
    RUNNING = 1
    STOPPED = 2

class P10V2(enum.Enum):
    RUNNING = 1
    STARTED = 1
    STOPPED = 2

print(P10V1.RUNNING.name, P10V2.STARTED.name)

STARTED RUNNING


**What the observation tells us.** A serializer that emits `.name` changes its output after a harmless-looking reorder; a value-based protocol does not change for these integers.

In [49]:
print('Names:', P10V1(1).name, P10V2(1).name)
print('Values:', P10V1(1).value, P10V2(1).value)

Names: STARTED RUNNING
Values: 1 1


### Step 2 — Construct the solution

Specify a version number plus a stable numeric code. Decode with an explicit version check and the target enum.

In [50]:
def p10_encode(member):
    return json.dumps({'schema': 1, 'code': member.value}, sort_keys=True)

def p10_decode(payload, cls):
    obj = json.loads(payload)
    if set(obj) != {'schema', 'code'} or obj['schema'] != 1:
        raise ValueError('unknown wire schema')
    if type(obj['code']) is not int:
        raise ValueError('integer code required')
    return cls(obj['code'])

p10_wire = p10_encode(P10V1.RUNNING)
print(p10_wire, p10_decode(p10_wire, P10V2))

{"code": 1, "schema": 1} P10V2.RUNNING


### Step 3 — Apply it to an edge case

Numeric codes are stable *only when maintained as stable protocol identifiers*. Changing numeric assignments would require another migration; the version field cannot fix arbitrary incompatible changes by itself.

In [51]:
for cls in [P10V1, P10V2]:
    print(cls.__name__, p10_encode(cls(1)), p10_decode(p10_wire, cls).name)

P10V1 {"code": 1, "schema": 1} STARTED
P10V2 {"code": 1, "schema": 1} RUNNING


### Full solution: reasoning and correctness

Round-trip code 1 to either release without assuming that `.name` remains unchanged; validate schema and types at the boundary.

**Verification.** The following cell uses assertions. If it runs without error, the stated behavior holds in this notebook environment.

In [52]:
assert p10_encode(P10V1.STARTED) == p10_encode(P10V2.RUNNING)
assert p10_decode(p10_wire, P10V1) is P10V1.STARTED
assert p10_decode(p10_wire, P10V2) is P10V2.RUNNING
for bad in ['{"schema": 99, "code": 1}', '{"schema": 1, "code": true}']:
    try:
        p10_decode(bad, P10V2)
    except ValueError:
        pass
    else:
        raise AssertionError('invalid wire message accepted')
print('P10 passed')

P10 passed


**Takeaway / further challenge.** Add version 2 with an explicit translation table if a code must be retired.

## Problem 11 — Explore copy and pickle without inventing alias provenance

**Scenario.** A job queue copies and serializes enum values before replay.

**Your task.** Test what survives copying and pickling; decide whether the original alias spelling can be reconstructed.

**How to work through this tutorial:** predict the result before running each investigation cell. Read the interpretation before moving to the implementation. The last code cell is an executable solution check.

### Step 1 — Identify the invariant

An alias reference is already the canonical member object before copying begins. Serialization cannot recover information that the object never held.

In [53]:
class P11Mode(enum.Enum):
    LIVE = 'L'
    ONLINE = 'L'
    OFFLINE = 'O'

p11_alias = P11Mode.ONLINE
print(p11_alias.name, p11_alias is P11Mode.LIVE)

LIVE True


**What the observation tells us.** Copy operations preserve enum identity. Even if a new wrapper object were created, it would have no stored record of the `ONLINE` input spelling.

In [54]:
print('copy:', copy.copy(p11_alias) is P11Mode.LIVE)
print('deepcopy:', copy.deepcopy(p11_alias) is P11Mode.LIVE)

copy: True
deepcopy: True


### Step 2 — Construct the solution

Round-trip with pickle, then design a separate record if audit provenance matters. Pickle is suitable only for trusted data; it is **not** a safe interchange format for untrusted input.

In [55]:
p11_roundtrip = pickle.loads(pickle.dumps(p11_alias))
print('Unpickled:', p11_roundtrip, 'original spelling:', p11_roundtrip.name)

p11_record = {'source_name': 'ONLINE', 'canonical_value': p11_alias.value}
print(p11_record)

Unpickled: P11Mode.LIVE original spelling: LIVE
{'source_name': 'ONLINE', 'canonical_value': 'L'}


### Step 3 — Apply it to an edge case

The wrapper restores both the semantic member and the original input name as separate fields when using a trusted in-process mapping.

In [56]:
p11_restored = (P11Mode(p11_record['canonical_value']),
                p11_record['source_name'])
print(p11_restored)

(<P11Mode.LIVE: 'L'>, 'ONLINE')


### Full solution: reasoning and correctness

Identity is retained across copy and standard pickle, but input spelling provenance needs explicit storage. Avoid `pickle.loads` on network messages or other untrusted bytes.

**Verification.** The following cell uses assertions. If it runs without error, the stated behavior holds in this notebook environment.

In [57]:
assert copy.copy(p11_alias) is P11Mode.LIVE
assert copy.deepcopy(p11_alias) is P11Mode.LIVE
assert pickle.loads(pickle.dumps(p11_alias)) is P11Mode.LIVE
assert p11_restored == (P11Mode.LIVE, 'ONLINE')
print('P11 passed')

P11 passed


**Takeaway / further challenge.** Use JSON with a validated schema to persist the wrapper for an untrusted-input boundary.

## Problem 12 — Build a migration that tolerates renamed canonical members

**Scenario.** A storage layer persisted canonical enum names in old records and a new release promotes an alias to canonical.

**Your task.** Write a decoder that accepts both historical and current names, then converts records into a stable internal code.

**How to work through this tutorial:** predict the result before running each investigation cell. Read the interpretation before moving to the implementation. The last code cell is an executable solution check.

### Step 1 — Identify the invariant

Compare old and new names without assuming that a name change implies a changed semantic value.

In [58]:
class P12Old(enum.Enum):
    OPENED = 1
    ACTIVE = 1
    FINISHED = 2

class P12New(enum.Enum):
    ACTIVE = 1
    OPENED = 1
    FINISHED = 2

p12_old_rows = [{'state': 'OPENED'}, {'state': 'FINISHED'}]
print('New canonical name for 1:', P12New(1).name)

New canonical name for 1: ACTIVE


**What the observation tells us.** Name lookup still accepts `OPENED` because it remains declared as an alias. A new serializer, however, will output `ACTIVE` for value 1.

In [59]:
print([P12New[row['state']] for row in p12_old_rows])

[<P12New.ACTIVE: 1>, <P12New.FINISHED: 2>]


### Step 2 — Construct the solution

Decode the historical field using the new enum’s accepted names, then persist a stable integer code. Validate the field shape instead of guessing from arbitrary strings.

In [60]:
def p12_migrate(row):
    if set(row) != {'state'} or not isinstance(row['state'], str):
        raise ValueError('expected one string field: state')
    try:
        member = P12New[row['state']]
    except KeyError as exc:
        raise ValueError('unknown historical state') from exc
    return {'schema': 2, 'code': member.value}

p12_migrated = [p12_migrate(row) for row in p12_old_rows]
print(p12_migrated)

[{'schema': 2, 'code': 1}, {'schema': 2, 'code': 2}]


### Step 3 — Apply it to an edge case

When a legacy alias is removed, a migration table is needed. Retaining aliases is one option; mapping removed historical names is another.

In [61]:
p12_removed_name_map = {'IN_PROGRESS': P12New.ACTIVE}
def p12_migrate_extended(row):
    if row.get('state') in p12_removed_name_map:
        return {'schema': 2, 'code': p12_removed_name_map[row['state']].value}
    return p12_migrate(row)
print(p12_migrate_extended({'state': 'IN_PROGRESS'}))

{'schema': 2, 'code': 1}


### Full solution: reasoning and correctness

The migration captures intended semantics in stable codes while accepting historical names. Name acceptance and name emission are two different contracts.

**Verification.** The following cell uses assertions. If it runs without error, the stated behavior holds in this notebook environment.

In [62]:
assert p12_migrate({'state': 'OPENED'}) == {'schema': 2, 'code': 1}
assert p12_migrate({'state': 'ACTIVE'}) == {'schema': 2, 'code': 1}
assert p12_migrate_extended({'state': 'IN_PROGRESS'})['code'] == 1
try:
    p12_migrate({'state': 'NOT_REAL'})
except ValueError:
    pass
else:
    raise AssertionError('unknown name accepted')
print('P12 passed')

P12 passed


**Takeaway / further challenge.** Write a migration report that counts deprecated input names without changing semantic state counts.

## Problem 13 — Round-trip an audit record with explicit provenance

**Scenario.** Investigators must distinguish which spelling a partner supplied, even after normalization to an enum.

**Your task.** Build a frozen value object, validate name-to-member consistency, and produce a JSON-safe record.

**How to work through this tutorial:** predict the result before running each investigation cell. Read the interpretation before moving to the implementation. The last code cell is an executable solution check.

### Step 1 — Identify the invariant

An alias’s `.name` is canonical, so assigning only the member to an audit record loses the original spelling.

In [63]:
class P13Outcome(enum.Enum):
    ACCEPTED = 'A'
    APPROVED = 'A'
    REJECTED = 'R'

print(P13Outcome.APPROVED.name)

ACCEPTED


**What the observation tells us.** An audit representation needs both the declared input spelling and the normalized member. Frozen dataclasses prevent accidental reassignment of those fields.

In [64]:
@dataclass(frozen=True)
class P13Audit:
    source_name: str
    outcome: P13Outcome

    def __post_init__(self):
        if self.source_name not in P13Outcome.__members__:
            raise ValueError('unrecognized source name')
        if P13Outcome[self.source_name] is not self.outcome:
            raise ValueError('source name and semantic outcome disagree')

record13 = P13Audit('APPROVED', P13Outcome.ACCEPTED)
print(record13)

P13Audit(source_name='APPROVED', outcome=<P13Outcome.ACCEPTED: 'A'>)


### Step 2 — Construct the solution

JSON needs primitive values. Export the original spelling and stable code, then reconstruct using validated name lookup.

In [65]:
def p13_to_json(record):
    return json.dumps({'source_name': record.source_name,
                       'outcome_code': record.outcome.value}, sort_keys=True)

def p13_from_json(payload):
    obj = json.loads(payload)
    if set(obj) != {'source_name', 'outcome_code'}:
        raise ValueError('unexpected record shape')
    name = obj['source_name']
    if not isinstance(name, str):
        raise ValueError('source name must be a string')
    return P13Audit(name, P13Outcome(obj['outcome_code']))

p13_wire = p13_to_json(record13)
print(p13_wire, p13_from_json(p13_wire))

{"outcome_code": "A", "source_name": "APPROVED"} P13Audit(source_name='APPROVED', outcome=<P13Outcome.ACCEPTED: 'A'>)


### Step 3 — Apply it to an edge case

Try a contradictory record to prove the two stored fields are not allowed to silently disagree.

In [66]:
try:
    p13_from_json('{"source_name":"APPROVED","outcome_code":"R"}')
except ValueError as exc:
    print('Rejected inconsistent data:', exc)

Rejected inconsistent data: source name and semantic outcome disagree


### Full solution: reasoning and correctness

The restored record preserves source provenance but also detects falsified or corrupted code/name combinations.

**Verification.** The following cell uses assertions. If it runs without error, the stated behavior holds in this notebook environment.

In [67]:
assert p13_from_json(p13_wire) == record13
assert p13_from_json(p13_wire).source_name == 'APPROVED'
assert p13_from_json(p13_wire).outcome is P13Outcome.ACCEPTED
try:
    P13Audit('APPROVED', P13Outcome.REJECTED)
except ValueError:
    pass
else:
    raise AssertionError('inconsistent record accepted')
print('P13 passed')

P13 passed


**Takeaway / further challenge.** Add a schema version and prove old records can be read by an explicit upgrader.

---

# Part IV — Defensive interfaces and surprising values

An enum is not automatically an input parser or a validation strategy. These problems make the boundary checks explicit.

## Problem 14 — Avoid source-policy ambiguity in a shared parser

**Scenario.** Two providers use the same raw token `ready` but have different semantics.

**Your task.** Implement a source-aware parser that rejects an unqualified token rather than guessing from a global alias map.

**How to work through this tutorial:** predict the result before running each investigation cell. Read the interpretation before moving to the implementation. The last code cell is an executable solution check.

### Step 1 — Identify the invariant

First model semantic states with distinct values, then examine provider-specific mappings.

In [68]:
@enum.unique
class P14Phase(enum.Enum):
    WAITING = 1
    OPERATING = 2
    FINISHED = 3

p14_maps = {
    'alpha': {'ready': P14Phase.WAITING, 'busy': P14Phase.OPERATING},
    'beta': {'ready': P14Phase.OPERATING, 'done': P14Phase.FINISHED},
}
print('alpha ready:', p14_maps['alpha']['ready'])
print('beta ready:', p14_maps['beta']['ready'])

alpha ready: P14Phase.WAITING
beta ready: P14Phase.OPERATING


**What the observation tells us.** Blindly flattening the providers’ dictionaries loses one meaning for `ready`. An enum alias cannot repair a missing source identifier.

In [69]:
p14_flat = {token: state for aliases in p14_maps.values()
            for token, state in aliases.items()}
print('Dangerous global ready:', p14_flat['ready'])

Dangerous global ready: P14Phase.OPERATING


### Step 2 — Construct the solution

Require an explicit provider and an exact token. Fail closed on unknown values.

In [70]:
def p14_parse(provider, token):
    if provider not in p14_maps:
        raise ValueError(f'unknown provider: {provider!r}')
    try:
        return p14_maps[provider][token]
    except KeyError as exc:
        raise ValueError(f'unknown {provider} token: {token!r}') from exc

print(p14_parse('alpha', 'ready'), p14_parse('beta', 'ready'))

P14Phase.WAITING P14Phase.OPERATING


### Step 3 — Apply it to an edge case

An audit trail may still need both provider and original token. Return them explicitly in a small record.

In [71]:
def p14_parse_record(provider, token):
    return {'provider': provider, 'source_token': token,
            'canonical': p14_parse(provider, token)}
print(p14_parse_record('alpha', 'busy'))

{'provider': 'alpha', 'source_token': 'busy', 'canonical': <P14Phase.OPERATING: 2>}


### Full solution: reasoning and correctness

Provider context determines how raw spelling maps to semantics. Neither `.name` nor enum aliases should be used to guess the origin.

**Verification.** The following cell uses assertions. If it runs without error, the stated behavior holds in this notebook environment.

In [72]:
assert p14_parse('alpha', 'ready') is P14Phase.WAITING
assert p14_parse('beta', 'ready') is P14Phase.OPERATING
for provider, token in [('unknown', 'ready'), ('alpha', 'done')]:
    try:
        p14_parse(provider, token)
    except ValueError:
        pass
    else:
        raise AssertionError('invalid provider/token was accepted')
print('P14 passed')

P14 passed


**Takeaway / further challenge.** Extend the parser to validate an explicit `provider_version` field as well.

## Problem 15 — Handle unhashable enum values safely

**Scenario.** A legacy enum stores dictionaries as values and two dictionary payloads compare equal.

**Your task.** Prove aliases can form even when values are unhashable; redesign the API around immutable stable identifiers.

**How to work through this tutorial:** predict the result before running each investigation cell. Read the interpretation before moving to the implementation. The last code cell is an executable solution check.

### Step 1 — Identify the invariant

Enum can construct members from some unhashable values such as dictionaries. Equality can still make those values aliases; a hash-based assumption is unsafe.

In [73]:
class P15Legacy(enum.Enum):
    RED = {'hex': '#f00'}
    CRIMSON = {'hex': '#f00'}
    BLUE = {'hex': '#00f'}

print(P15Legacy.CRIMSON is P15Legacy.RED)
print(P15Legacy({'hex': '#f00'}))

True
P15Legacy.RED


**What the observation tells us.** The dictionary values are mutable. Editing the stored dictionary risks invalidating semantic assumptions, especially if another component relies on it for lookups.

In [74]:
print('Original legacy value:', P15Legacy.RED.value)
print('Canonical members:', [m.name for m in P15Legacy])

Original legacy value: {'hex': '#f00'}
Canonical members: ['RED', 'BLUE']


### Step 2 — Construct the solution

Use immutable codes for identity and store rich metadata in a separate mapping. A mapping can still be mutable, but its mutation no longer changes enum identity.

In [75]:
@enum.unique
class P15Color(enum.Enum):
    RED = 'red'
    BLUE = 'blue'

p15_metadata = {P15Color.RED: {'hex': '#f00'},
                P15Color.BLUE: {'hex': '#00f'}}
p15_alias_names = {'CRIMSON': P15Color.RED}
print(P15Color.RED.value, p15_metadata[p15_alias_names['CRIMSON']])

red {'hex': '#f00'}


### Step 3 — Apply it to an edge case

Forbid mutation of the identity code itself and be explicit about who owns the metadata. Copying on read prevents callers from changing the registry accidentally.

In [76]:
def p15_get_metadata(color):
    return dict(p15_metadata[color])

p15_copy = p15_get_metadata(P15Color.RED)
p15_copy['hex'] = 'changed only in local copy'
print('Registry untouched:', p15_metadata[P15Color.RED])

Registry untouched: {'hex': '#f00'}


### Full solution: reasoning and correctness

Equality of unhashable values is enough to create aliases, but immutable codes and copied metadata make a safer public interface.

**Verification.** The following cell uses assertions. If it runs without error, the stated behavior holds in this notebook environment.

In [77]:
assert P15Legacy.CRIMSON is P15Legacy.RED
assert P15Legacy({'hex': '#f00'}) is P15Legacy.RED
assert p15_metadata[P15Color.RED]['hex'] == '#f00'
assert len(P15Color) == 2
print('P15 passed')

P15 passed


**Takeaway / further challenge.** Use a frozen dataclass as metadata and explain which invariants it enforces.

## Problem 16 — Warn on deprecated aliases at the input boundary

**Scenario.** An API accepts old member names but wants to warn callers when they use deprecated spellings.

**Your task.** Build a parser that emits a targeted warning on a deprecated *name*; explain why warning on the enum member itself would be incorrect.

**How to work through this tutorial:** predict the result before running each investigation cell. Read the interpretation before moving to the implementation. The last code cell is an executable solution check.

### Step 1 — Identify the invariant

By the time a caller has an enum member, the original alias spelling is gone. Deprecation based on spelling must happen before that collapse.

In [78]:
class P16Level(enum.Enum):
    NORMAL = 1
    STANDARD = 1
    HIGH = 2

print(P16Level.STANDARD.name, P16Level.STANDARD is P16Level.NORMAL)

NORMAL True


**What the observation tells us.** A check such as `if member is P16Level.STANDARD` is also true for `P16Level.NORMAL`. That cannot identify who used the old spelling.

In [79]:
print('Indistinguishable enum members:', P16Level.NORMAL is P16Level.STANDARD)

Indistinguishable enum members: True


### Step 2 — Construct the solution

Check the raw name, warn explicitly, then use bracket lookup. The warning category is suitable for a deprecated interface; no warning is emitted for canonical names.

In [80]:
p16_deprecated = {'STANDARD': 'Use NORMAL instead'}

def p16_parse_name(name):
    if name not in P16Level.__members__:
        raise ValueError(f'unknown level: {name}')
    if name in p16_deprecated:
        warnings.warn(p16_deprecated[name], DeprecationWarning, stacklevel=2)
    return P16Level[name]

with warnings.catch_warnings(record=True) as p16_caught:
    warnings.simplefilter('always', DeprecationWarning)
    p16_result = p16_parse_name('STANDARD')
print(p16_result, [str(w.message) for w in p16_caught])

P16Level.NORMAL ['Use NORMAL instead']


### Step 3 — Apply it to an edge case

Confirm that canonical input does not produce a deprecation warning even though it resolves to exactly the same member.

In [81]:
with warnings.catch_warnings(record=True) as p16_current:
    warnings.simplefilter('always', DeprecationWarning)
    p16_parse_name('NORMAL')
print('Current-name warnings:', len(p16_current))

Current-name warnings: 0


### Full solution: reasoning and correctness

Deprecation is a property of a spelling/interface, not of the underlying semantic member. Keep policy at the parsing boundary.

**Verification.** The following cell uses assertions. If it runs without error, the stated behavior holds in this notebook environment.

In [82]:
assert p16_result is P16Level.NORMAL
assert len(p16_caught) == 1
assert p16_caught[0].category is DeprecationWarning
assert len(p16_current) == 0
try:
    p16_parse_name('UNRECOGNIZED')
except ValueError:
    pass
else:
    raise AssertionError('unknown name accepted')
print('P16 passed')

P16 passed


**Takeaway / further challenge.** Add an announced removal version to the warning message without changing the enum definition.

## Problem 17 — Generate a schema that distinguishes accepted names from states

**Scenario.** A documentation generator wants to publish all accepted member names and all unique semantic values.

**Your task.** Produce a deterministic schema that contains both lists without pretending every alias is an independent value.

**How to work through this tutorial:** predict the result before running each investigation cell. Read the interpretation before moving to the implementation. The last code cell is an executable solution check.

### Step 1 — Identify the invariant

A one-dimensional `list(EnumClass)` enumerates canonical members only. A name-based parser, however, accepts names from `__members__`.

In [83]:
class P17Access(enum.Enum):
    READ = 'r'
    VIEW = 'r'
    WRITE = 'w'
    EDIT = 'w'
    ADMIN = 'a'

print('Canonical:', [m.name for m in P17Access])
print('Accepted:', list(P17Access.__members__))

Canonical: ['READ', 'WRITE', 'ADMIN']
Accepted: ['READ', 'VIEW', 'WRITE', 'EDIT', 'ADMIN']


**What the observation tells us.** Separating semantic values and accepted spellings avoids under-documenting the parser or overcounting distinct states.

In [84]:
print('Counts:', len(P17Access), len(P17Access.__members__))

Counts: 3 5


### Step 2 — Construct the solution

Publish two explicit structures: accepted names with their canonical targets, and canonical values with their stable wire codes.

In [85]:
def p17_schema(cls):
    return {
        'accepted_names': {name: member.name for name, member
                           in cls.__members__.items()},
        'canonical_values': {member.name: member.value for member in cls},
    }

p17_doc = p17_schema(P17Access)
print(json.dumps(p17_doc, indent=2))

{
  "accepted_names": {
    "READ": "READ",
    "VIEW": "READ",
    "WRITE": "WRITE",
    "EDIT": "WRITE",
    "ADMIN": "ADMIN"
  },
  "canonical_values": {
    "READ": "r",
    "WRITE": "w",
    "ADMIN": "a"
  }
}


### Step 3 — Apply it to an edge case

A consumer can now display canonical states but still validate historic names. Neither list requires guessing from enum attributes.

In [86]:
def p17_validate_documented_name(name, schema):
    if name not in schema['accepted_names']:
        raise ValueError(f'name not documented: {name}')
    return schema['accepted_names'][name]
print('EDIT maps to', p17_validate_documented_name('EDIT', p17_doc))

EDIT maps to WRITE


### Full solution: reasoning and correctness

An enum exposes two valid but different views. Schema generation should label them so downstream tools do not confuse names, objects, and wire values.

**Verification.** The following cell uses assertions. If it runs without error, the stated behavior holds in this notebook environment.

In [87]:
assert p17_doc['accepted_names']['VIEW'] == 'READ'
assert p17_doc['accepted_names']['EDIT'] == 'WRITE'
assert len(p17_doc['canonical_values']) == len(P17Access) == 3
assert len(p17_doc['accepted_names']) == len(P17Access.__members__) == 5
print('P17 passed')

P17 passed


**Takeaway / further challenge.** Add a third structure for deprecated alias names so generated documentation can mark them explicitly.

---

# Part V — Capstone

The final problem integrates construction, validation, parsing, auditing, serialization, and regression tests in a small, dependency-free ingestion service.

## Problem 18 — Build an alias-aware ingestion boundary end to end

**Scenario.** A service reads versioned partner messages and must normalize statuses, audit raw spellings, reject malformed data, and emit stable wire codes.

**Your task.** Implement a small but complete boundary with a unique semantic enum, source-specific token maps, a frozen record, and a strict export function.

**How to work through this tutorial:** predict the result before running each investigation cell. Read the interpretation before moving to the implementation. The last code cell is an executable solution check.

### Step 1 — Identify the invariant

The semantic state definition is deliberately alias-free: aliases belong to the accepted input vocabulary here, which changes independently by partner and version.

In [88]:
@enum.unique
class P18State(enum.Enum):
    WAITING = 10
    RUNNING = 20
    DONE = 30
    ERROR = 40

p18_vocab = {
    ('warehouse', 1): {'queued': P18State.WAITING,
                       'busy': P18State.RUNNING,
                       'ok': P18State.DONE,
                       'bad': P18State.ERROR},
    ('warehouse', 2): {'waiting': P18State.WAITING,
                       'running': P18State.RUNNING,
                       'done': P18State.DONE,
                       'error': P18State.ERROR},
    ('billing', 1): {'hold': P18State.WAITING,
                     'working': P18State.RUNNING,
                     'paid': P18State.DONE,
                     'declined': P18State.ERROR},
}
print('Internal states:', [s.name for s in P18State])

Internal states: ['WAITING', 'RUNNING', 'DONE', 'ERROR']


**What the observation tells us.** The vocabulary is keyed by both provider and version. A token cannot accidentally acquire a meaning from the wrong provider.

In [89]:
for key, tokens in p18_vocab.items():
    print(key, 'accepts', sorted(tokens))

('warehouse', 1) accepts ['bad', 'busy', 'ok', 'queued']
('warehouse', 2) accepts ['done', 'error', 'running', 'waiting']
('billing', 1) accepts ['declined', 'hold', 'paid', 'working']


### Step 2 — Construct the solution

Build a strict decoder. Validate the message shape and types, use an explicit vocabulary, and preserve exact provenance in an immutable object.

In [90]:
@dataclass(frozen=True)
class P18Record:
    provider: str
    version: int
    source_token: str
    state: P18State


def p18_ingest(message: Mapping):
    if not isinstance(message, dict) or set(message) != {'provider', 'version', 'status'}:
        raise ValueError('expected exactly provider, version, status')
    provider, version, token = message['provider'], message['version'], message['status']
    if not isinstance(provider, str) or type(version) is not int or not isinstance(token, str):
        raise ValueError('invalid field types')
    vocabulary = p18_vocab.get((provider, version))
    if vocabulary is None or token not in vocabulary:
        raise ValueError('unknown provider/version/status combination')
    return P18Record(provider, version, token, vocabulary[token])

p18_sample = p18_ingest({'provider': 'warehouse', 'version': 1, 'status': 'busy'})
print(p18_sample)

P18Record(provider='warehouse', version=1, source_token='busy', state=<P18State.RUNNING: 20>)


### Step 3 — Apply it to an edge case

Export stable machine fields separately from audit fields. Include only JSON-compatible primitives and preserve the raw token for forensics.

In [91]:
def p18_export(record):
    return json.dumps({
        'schema': 1,
        'state_code': record.state.value,
        'state_name': record.state.name,
        'audit': {'provider': record.provider,
                  'version': record.version,
                  'raw_status': record.source_token},
    }, sort_keys=True)

p18_messages = [
    {'provider': 'warehouse', 'version': 1, 'status': 'queued'},
    {'provider': 'warehouse', 'version': 2, 'status': 'running'},
    {'provider': 'billing', 'version': 1, 'status': 'paid'},
]
p18_results = [p18_ingest(row) for row in p18_messages]
for record in p18_results:
    print(p18_export(record))

{"audit": {"provider": "warehouse", "raw_status": "queued", "version": 1}, "schema": 1, "state_code": 10, "state_name": "WAITING"}
{"audit": {"provider": "warehouse", "raw_status": "running", "version": 2}, "schema": 1, "state_code": 20, "state_name": "RUNNING"}
{"audit": {"provider": "billing", "raw_status": "paid", "version": 1}, "schema": 1, "state_code": 30, "state_name": "DONE"}


### Step 4 — Regression and adverse-input testing

Run positive and negative tests. The tests check all registered tokens, type failures, unknown input, and one equivalent semantic state across two partner versions.

In [92]:
for (provider, version), vocab in p18_vocab.items():
    for token, member in vocab.items():
        result = p18_ingest({'provider': provider, 'version': version, 'status': token})
        assert result.state is member
        assert result.source_token == token
        assert json.loads(p18_export(result))['state_code'] == member.value

p18_invalid = [
    {'provider': 'warehouse', 'version': True, 'status': 'busy'},
    {'provider': 'warehouse', 'version': 1, 'status': 'running'},
    {'provider': 'billing', 'version': 9, 'status': 'paid'},
    {'provider': 'billing', 'version': 1, 'status': 'paid', 'debug': True},
]
for message in p18_invalid:
    try:
        p18_ingest(message)
    except ValueError:
        pass
    else:
        raise AssertionError(f'invalid message accepted: {message}')
print('Positive and negative cases checked')

Positive and negative cases checked


### Full solution: reasoning and correctness

The boundary is explicit at every stage: unique internal members, versioned input vocabulary, strict parsing, preserved provenance, and stable serialization. An alias cannot secretly override a handler, change a partner’s meaning, or erase an audit token.

**Verification.** The following cell uses assertions. If it runs without error, the stated behavior holds in this notebook environment.

In [93]:
assert len(P18State) == len(P18State.__members__) == 4
assert p18_ingest({'provider': 'warehouse', 'version': 1, 'status': 'busy'}).state is P18State.RUNNING
assert p18_ingest({'provider': 'warehouse', 'version': 2, 'status': 'running'}).state is P18State.RUNNING
assert p18_sample.source_token == 'busy'
assert json.loads(p18_export(p18_sample))['audit']['raw_status'] == 'busy'
assert json.loads(p18_export(p18_sample))['state_code'] == 20
assert len(p18_results) == 3
print('P18 passed')

P18 passed


**Takeaway / further challenge.** Implement a new partner version by adding one vocabulary entry, without editing the enum or changing existing wire codes.

---

# Final synthesis — checklist for production code

**Use enum aliases when** distinct Python names genuinely designate the *same semantic member*, and losing the spelling is acceptable.

**Use an external mapping when** input varies by partner, version, case, punctuation, or deprecation policy; when two names carry different metadata; or when you must reconstruct the exact original token.

**Use `@enum.unique` when** each declared member is meant to describe a genuinely distinct state, operation, or event. It raises `ValueError` during class creation if duplicate values produce aliases.

**Inspect both views:** `list(MyEnum)` shows distinct members; `MyEnum.__members__` includes all declared names in declaration order. `MyEnum['NAME']` is name lookup; `MyEnum(value)` is value lookup. These two paths are not interchangeable.

**Protect boundaries:** validate external data before discarding names; avoid `.name` as a stable wire format unless the canonical-name contract is intentionally fixed; avoid untrusted pickle; test ambiguity, duplicate registrations, reordering, and unknown inputs.

The assertions below provide a final summary of the worked exercises. They complement, but do not replace, your own business requirements and integration tests.

In [94]:
p_final_expected = [
    P01Old(10).name == 'ECONOMY',
    P02Flawed.WAITING is P02Flawed.QUEUED,
    P03Packet.ADD is P03Packet.CREATE,
    len(P04Fixed) == 4,
    len(P05Compatible.__members__) == 3,
    len(p06_good) == 1,
    p07_route_token('OK')['action'] == 'acknowledge',
    len(p08_edges) == 4,
    P09Event.CAPTURE is not P09Event.AUTHORIZE,
    p10_decode(p10_wire, P10V2) is P10V2.RUNNING,
    p11_restored[1] == 'ONLINE',
    p12_migrate({'state': 'OPENED'})['code'] == 1,
    p13_from_json(p13_wire) == record13,
    p14_parse('beta', 'ready') is P14Phase.OPERATING,
    P15Legacy.CRIMSON is P15Legacy.RED,
    len(p16_caught) == 1,
    len(p17_doc['accepted_names']) == 5,
    len(p18_results) == 3,
]
assert all(p_final_expected)
print(f'Final integration checks passed: {sum(p_final_expected)}/{len(p_final_expected)} problems')

Final integration checks passed: 18/18 problems


## Self-study prompts (no extra dependencies)

1. Pick one input parser and add a formal distinction between *unknown name*, *deprecated name*, and *known name with disallowed source*.
2. Derive a compatibility diff between two enum definitions that reports changed values, removed names, and changed canonical names.
3. Add an audit round trip to the capstone: decode the exported JSON back to a validated `P18Record` and reject mismatched source tokens versus state codes.
4. Write a property-style loop over every accepted name and verify that name lookup always resolves to a canonical member encountered in iteration.

**End of notebook.**